<a href="https://colab.research.google.com/github/katumihamada0728/otobunkatsu/blob/main/%E3%83%89%E3%83%A9%E3%82%A4%E3%83%96%E3%81%8B%E3%82%89%E5%8F%96%E8%BE%BC%E3%82%92%E3%81%97%E3%81%A6%E9%9F%B3%E5%A3%B0%E5%88%86%E5%89%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

音声処理に必要な pydub と ffmpeg をインストールします。

In [ ]:
!apt-get install ffmpeg -y
!pip install pydub

Googleドライブとの連携認証を行います。実行するとアクセス許可のポップアップ（またはURL）が表示されるので、「Googleドライブに接続」を許可してください。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

約20分（1,200,000ミリ秒）を基本単位とし、目標時間の前後（検索ウィンドウ）で最も音量が小さい箇所を探して分割します。

In [ ]:
import os
from pydub import AudioSegment

# --- 設定部分 ---
input_path = "/content/drive/MyDrive/250106_知事年頭記者会見 (mp3cut).mp3"  # 元ファイル
output_dir = "/content/drive/MyDrive/split_smart/"  # 保存先フォルダ

target_length_ms = 20 * 60 * 1000  # 目標の長さ（20分）
search_window_ms = (
    3 * 60 * 1000
)  # 無音を探す範囲（目標時間の前後3分間 = 17分〜23分を検索）
step_ms = 200  # 判定単位（ミリ秒）
# ----------------

os.makedirs(output_dir, exist_ok=True)

print("音声ファイルを読み込んでいます...")
audio = AudioSegment.from_file(input_path, format="mp3")
total_length = len(audio)


# 指定範囲の中で最も音量が小さい（静かな）ポイントを探す関数
def find_quietest_point(audio_segment, search_start, search_end):
    best_point = search_start
    min_loudness = float("inf")

    # 指定範囲をスライドしながら平均音量(dBFS)を比較
    for current_pos in range(search_start, search_end, step_ms):
        # 0.5秒間のサンプル区間の音量を計測
        sample = audio_segment[current_pos : current_pos + 500]
        loudness = sample.dBFS

        if loudness < min_loudness:
            min_loudness = loudness
            best_point = current_pos

    return best_point


current_start = 0
chunk_count = 1
base_name = os.path.splitext(os.path.basename(input_path))[0]

print("スマート分割処理を開始します...")

while current_start < total_length:
    # 残り時間が「目標時間 + 検索範囲」より短ければそのまま最後まで切り出し
    if current_start + target_length_ms + (search_window_ms // 2) >= total_length:
        split_point = total_length
    else:
        # 理想のカット位置（20分地点）
        ideal_split = current_start + target_length_ms

        # 検索範囲の決定（理想位置の前後3分間）
        search_start = max(current_start, ideal_split - (search_window_ms // 2))
        search_end = min(total_length, ideal_split + (search_window_ms // 2))

        # 最も静かなポイントを選択
        split_point = find_quietest_point(audio, search_start, search_end)

    # 音声の切り出しと出力
    chunk = audio[current_start:split_point]
    output_filename = os.path.join(
        output_dir, f"{base_name}_part{chunk_count:02d}.mp3"
    )
    chunk.export(output_filename, format="mp3")

    start_min = round(current_start / 1000 / 60, 1)
    end_min = round(split_point / 1000 / 60, 1)
    duration_min = round((split_point - current_start) / 1000 / 60, 1)

    print(
        f"保存完了: {output_filename} ({start_min}分 〜 {end_min}分 | 長さ: {duration_min}分)"
    )

    current_start = split_point
    chunk_count += 1

print("\nすべてのスマート分割処理が完了しました！")